# Day 38 · OAuth 与应用骨架

**配套讲义**: [`days/day-38.md`](../days/day-38.md) ｜ **本地可跑，不需要 GPU**

把 OAuth 安装流程跑通：`/auth` → 用户授权 → 回调 → 换 access token → 存库；并说清 **HMAC 校验** 和 **session token** 的关系。

> 📌 本 notebook 由 `scripts/gen_days.py` 生成 —— **别手改**，
> 要改内容请改 `scripts/daygen/w7.py` 后重跑脚本。

## 0. 环境检查

In [ ]:
import sys
print("python:", sys.version.split()[0])
for m in ("numpy", "PIL", "yaml", "pandas"):
    try:
        mod = __import__(m)
        print(f"  {m:7s} {getattr(mod, '__version__', 'ok')}")
    except ImportError:
        print(f"  {m:7s} ❌ 缺 → pip install {m}")
print("\n→ 本机没 GPU 不影响今天：今天只用纯 Python / numpy")

## 1. HMAC 自检（两个坑都要拦住）

In [ ]:
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "src.shopify.auth"],
                   capture_output=True, text=True, cwd="..")
print(r.stdout or r.stderr)

## 2. 亲手踩一次坑

先写「错误」的校验逻辑，看它为什么失败 —— 比直接看对的实现印象深十倍。

In [ ]:
import hmac, hashlib
from urllib.parse import parse_qsl

secret = "test_secret_abc"
qs = "host=YWRtaW4&shop=myshop.myshopify.com&code=abc123&timestamp=1700000000&state=xyz"

# 正确做法：剔除 hmac（这里没有）→ 按 key 排序 → 拼接
params = dict(parse_qsl(qs))
correct_msg = "&".join(f"{k}={v}" for k, v in sorted(params.items()))
print("正确拼接:", correct_msg)

# 错误做法 1：直接用原始 query string
wrong_msg = qs
print("错误拼接:", wrong_msg)
print("两者相同吗:", correct_msg == wrong_msg, "← 顺序不同就会不同，所以排序是必须的")

h_correct = hmac.new(secret.encode(), correct_msg.encode(), hashlib.sha256).hexdigest()
h_wrong   = hmac.new(secret.encode(), wrong_msg.encode(), hashlib.sha256).hexdigest()
print("\n两种算法得到的签名相同吗:", h_correct == h_wrong, "（不相同 → 用错方法一定验签失败）")

## 3. SSRF 演示

In [ ]:
import sys; sys.path.insert(0, "..")
from src.shopify.auth import validate_shop

for domain in ["demo.myshopify.com", "evil.com",
               "demo.myshopify.com.evil.io", "127.0.0.1", "myshop.myshopify.com:8080"]:
    try:
        ok = validate_shop(domain)
    except Exception as e:
        ok = f"拒绝 ({type(e).__name__})"
    print(f"{domain:32s} → {ok}")

## 验收清单

- [ ] `python -m src.shopify.auth` 自检全部通过，**两个 HMAC 坑都被拦住**
- [ ] 能在测试店完成一次完整安装（`/auth` → 授权 → 回前台正常）
- [ ] 能解释 HMAC 校验和 session token 的关系（谁在前、各自防什么）
- [ ] 知道 SSRF 是什么、`validate_shop` 拦的是哪种攻击

**卡住了？** 回看 [`days/day-38.md`](../days/day-38.md) 第五节「容易踩的坑」。

> **明天**：`days/day-39.md` —— 店铺前台挂件（能让店主看见的那个部分）